# 02. 기술 지표 분석 (Technical Indicators)

수집된 자산 가격 데이터에 기술 지표를 적용하여 트렌드와 변동성을 분석한다.

**지표**: SMA, EMA, 볼린저 밴드, 롤링 변동성, Rate of Change

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

from db import load_prices, get_asset_list

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='whitegrid')

In [ ]:
# 자산 목록 확인
assets = get_asset_list()
assets

In [ ]:
def resample_prices(df, freq='1h'):
    """불규칙 수집 데이터를 정규 간격으로 리샘플링한다."""
    df = df.set_index('collected_at').sort_index()
    resampled = df['price'].resample(freq).last().ffill()
    return resampled.to_frame().reset_index()

def load_and_resample(symbol, source, freq='1h'):
    """자산 데이터를 로드하고 리샘플링한다."""
    df = load_prices(symbol=symbol, source=source)
    if df.empty:
        return df
    return resample_prices(df, freq)

## 1. 이동평균 (SMA vs EMA)

In [ ]:
WINDOWS = [5, 10, 20, 50]

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    df = load_and_resample(symbol, source)
    if df.empty or len(df) < max(WINDOWS):
        print(f"{source}:{symbol} — 데이터 부족, 건너뜀")
        continue

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['collected_at'], y=df['price'],
                             name='가격', line=dict(color='black', width=1)))

    colors_sma = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    colors_ema = ['#aec7e8', '#ffbb78', '#98df8a', '#ff9896']

    for w, c_sma, c_ema in zip(WINDOWS, colors_sma, colors_ema):
        sma = df['price'].rolling(window=w).mean()
        ema = df['price'].ewm(span=w, adjust=False).mean()
        fig.add_trace(go.Scatter(x=df['collected_at'], y=sma,
                                 name=f'SMA-{w}', line=dict(color=c_sma, dash='solid')))
        fig.add_trace(go.Scatter(x=df['collected_at'], y=ema,
                                 name=f'EMA-{w}', line=dict(color=c_ema, dash='dot')))

    fig.update_layout(title=f'{source}:{symbol} — SMA vs EMA 비교',
                      xaxis_title='시간', yaxis_title='가격',
                      height=500, hovermode='x unified')
    fig.show()

## 2. 볼린저 밴드 (Bollinger Bands)

In [ ]:
BB_WINDOW = 20
BB_STD = 2

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    df = load_and_resample(symbol, source)
    if df.empty or len(df) < BB_WINDOW:
        continue

    sma = df['price'].rolling(BB_WINDOW).mean()
    std = df['price'].rolling(BB_WINDOW).std()
    upper = sma + BB_STD * std
    lower = sma - BB_STD * std

    fig = go.Figure()

    # 밴드 영역
    fig.add_trace(go.Scatter(x=df['collected_at'], y=upper, name='상단 밴드',
                             line=dict(color='rgba(68,68,68,0.3)')))
    fig.add_trace(go.Scatter(x=df['collected_at'], y=lower, name='하단 밴드',
                             line=dict(color='rgba(68,68,68,0.3)'),
                             fill='tonexty', fillcolor='rgba(68,68,68,0.1)'))

    # 가격 + SMA
    fig.add_trace(go.Scatter(x=df['collected_at'], y=df['price'],
                             name='가격', line=dict(color='#1f77b4', width=1.5)))
    fig.add_trace(go.Scatter(x=df['collected_at'], y=sma,
                             name=f'SMA-{BB_WINDOW}', line=dict(color='orange', dash='dash')))

    # 밴드 이탈 표시
    above = df['price'] > upper
    below = df['price'] < lower
    if above.any():
        fig.add_trace(go.Scatter(x=df.loc[above, 'collected_at'], y=df.loc[above, 'price'],
                                 mode='markers', name='상단 이탈',
                                 marker=dict(color='red', size=6)))
    if below.any():
        fig.add_trace(go.Scatter(x=df.loc[below, 'collected_at'], y=df.loc[below, 'price'],
                                 mode='markers', name='하단 이탈',
                                 marker=dict(color='green', size=6)))

    fig.update_layout(title=f'{source}:{symbol} — 볼린저 밴드 ({BB_WINDOW}, {BB_STD}σ)',
                      xaxis_title='시간', yaxis_title='가격',
                      height=500, hovermode='x unified')
    fig.show()

## 3. 롤링 변동성 (Rolling Volatility)

In [ ]:
VOL_WINDOWS = [10, 20, 50]

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    df = load_and_resample(symbol, source)
    if df.empty or len(df) < max(VOL_WINDOWS):
        continue

    returns = df['price'].pct_change().dropna()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=[f'{source}:{symbol} 가격', '롤링 변동성 (수익률 표준편차)'],
                        row_heights=[0.4, 0.6], vertical_spacing=0.08)

    fig.add_trace(go.Scatter(x=df['collected_at'], y=df['price'],
                             name='가격', line=dict(color='black', width=1)), row=1, col=1)

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for w, c in zip(VOL_WINDOWS, colors):
        vol = returns.rolling(w).std()
        fig.add_trace(go.Scatter(x=df['collected_at'].iloc[1:], y=vol,
                                 name=f'변동성-{w}', line=dict(color=c)), row=2, col=1)

    fig.update_layout(height=600, hovermode='x unified',
                      title=f'{source}:{symbol} — 롤링 변동성')
    fig.show()

## 4. Rate of Change (ROC)

In [ ]:
ROC_WINDOWS = [5, 10, 20]

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    df = load_and_resample(symbol, source)
    if df.empty or len(df) < max(ROC_WINDOWS):
        continue

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=[f'{source}:{symbol} 가격', 'Rate of Change (%)'],
                        row_heights=[0.4, 0.6], vertical_spacing=0.08)

    fig.add_trace(go.Scatter(x=df['collected_at'], y=df['price'],
                             name='가격', line=dict(color='black', width=1)), row=1, col=1)

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for w, c in zip(ROC_WINDOWS, colors):
        roc = (df['price'] - df['price'].shift(w)) / df['price'].shift(w) * 100
        fig.add_trace(go.Scatter(x=df['collected_at'], y=roc,
                                 name=f'ROC-{w}', line=dict(color=c)), row=2, col=1)

    # 0선
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

    fig.update_layout(height=600, hovermode='x unified',
                      title=f'{source}:{symbol} — Rate of Change')
    fig.show()

## 5. 지표 요약 테이블

In [ ]:
# 최신 지표 값 종합
summary = []

for _, row in assets.iterrows():
    source, symbol, currency = row['source'], row['symbol'], row['quote_currency']
    df = load_and_resample(symbol, source)
    if df.empty or len(df) < 50:
        continue

    price = df['price'].iloc[-1]
    returns = df['price'].pct_change().dropna()

    sma_20 = df['price'].rolling(20).mean().iloc[-1]
    ema_20 = df['price'].ewm(span=20, adjust=False).mean().iloc[-1]
    bb_std = df['price'].rolling(20).std().iloc[-1]
    bb_upper = sma_20 + 2 * bb_std
    bb_lower = sma_20 - 2 * bb_std
    vol_20 = returns.rolling(20).std().iloc[-1]
    roc_10 = (price - df['price'].iloc[-11]) / df['price'].iloc[-11] * 100

    # 볼린저 위치: 밴드 내 백분위 (0% = 하단, 100% = 상단)
    bb_position = (price - bb_lower) / (bb_upper - bb_lower) * 100 if bb_upper != bb_lower else 50

    summary.append({
        '자산': f'{source}:{symbol}',
        '통화': currency,
        '현재가': f'{price:,.2f}',
        'SMA-20': f'{sma_20:,.2f}',
        'EMA-20': f'{ema_20:,.2f}',
        'BB 상단': f'{bb_upper:,.2f}',
        'BB 하단': f'{bb_lower:,.2f}',
        'BB 위치': f'{bb_position:.1f}%',
        '변동성-20': f'{vol_20:.6f}',
        'ROC-10': f'{roc_10:.2f}%'
    })

summary_df = pd.DataFrame(summary)
summary_df.style.set_caption('최신 기술 지표 요약')